# 01 - Train a custom baseline (U-Net / MedNeXt / DA-VNet / AKS-UNet)

Runtime -> Change runtime type -> **T4 GPU**. Each team member sets `MODEL` to their model.
Training checkpoints to Drive every epoch; if Colab disconnects or units run out, **just re-run all cells** and it
resumes. Set `MAX_HOURS` a little below your remaining session so it stops cleanly.

In [ ]:
MODEL = 'mednext'      # 'unet3d' | 'mednext' | 'davnet' | 'aksunet'
EPOCHS = 200           # x 250 iterations; capped for benchmarking, keep identical across models
MAX_HOURS = 3.5
PATCH = '192 192 16'   # H W D(slices). On OOM: '160 160 16' or BATCH = 1
BATCH = 2

In [ ]:
# --- shared setup: Drive + code ---------------------------------------------
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive/figo'          # shared project folder (one Colab account)
REPO_URL = 'https://github.com/DeekshaBandi/test.git'  # private repo? use https://<token>@github.com/...
REPO = '/content/figo-repo'

import os, subprocess
os.makedirs(DRIVE, exist_ok=True)
if not os.path.exists(REPO):
    subprocess.run(['git', 'clone', '-q', REPO_URL, REPO], check=True)
else:
    subprocess.run(['git', '-C', REPO, 'pull', '-q'], check=True)
%cd {REPO}
!pip -q install -r requirements.txt

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
# copy the cache to local disk once per session (fast random access during training)
!rsync -a {DRIVE}/data/cache_0.9mm/ /content/cache/
RUN = f'{DRIVE}/runs/{MODEL}'

In [ ]:
!python -m figo.train --model {MODEL} --cache-dir /content/cache --splits {DRIVE}/data/splits.json \
    --out-dir {RUN} --epochs {EPOCHS} --max-hours {MAX_HOURS} --patch-size {PATCH} --batch-size {BATCH}

In [ ]:
# learning curves
import csv, matplotlib.pyplot as plt
log = list(csv.DictReader(open(f'{RUN}/log.csv')))
plt.plot([float(r['train_loss']) for r in log], label='train loss')
v = [(int(r['epoch']), float(r['val_mean'])) for r in log if r['val_mean']]
if v: plt.plot(*zip(*v), 'o-', label='val mean Dice')
plt.legend(); plt.xlabel('epoch'); plt.show()

## Test-set inference and evaluation (run after training has finished)

In [ ]:
PRED = f'{DRIVE}/preds/{MODEL}'
RES = f'{DRIVE}/results'
!python -m figo.infer --checkpoint {RUN}/best.pt --data-dir {DRIVE}/data/clean --splits {DRIVE}/data/splits.json \
    --subset test --out-dir {PRED}
!python -m figo.evaluate --pred-dir {PRED} --data-dir {DRIVE}/data/clean --splits {DRIVE}/data/splits.json \
    --subset test --out {RES}/{MODEL}_seg.csv
!python -m figo.figo_eval agreement --ref-dir {DRIVE}/data/clean --pred-dir {PRED} \
    --splits {DRIVE}/data/splits.json --subset test --out {RES}/{MODEL}_figo_agreement.csv